In [1]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.1/985.1 kB 29.6 MB/s eta 0:00:00


In [8]:
!pip install scikit-learn==1.2.2
import sklearn
import pypfopt
print(sklearn.__version__)
print(pypfopt.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 95.8 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.5.2
    Uninstalling scikit-learn-1.5.2:
      Successfully uninstalled scikit-learn-1.5.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlxtend 0.23.3 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.
1.2.2
1.5.5


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy

In [10]:
!pip install yfinance==0.2.40
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.50
    Uninstalling yfinance-0.2.50:
      Successfully uninstalled yfinance-0.2.50
yfinance: pandas_datareader support is deprecated & semi-broken so will be removed in a future verison. Just use yfinance.


In [11]:
def modify_date(date):
    return date[:10]

In [12]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [13]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [14]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [15]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [16]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [17]:
def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

In [151]:
# Preselection parameters set up
method = "K_means_multiple"          # or avg or anyother method
stock_type = "S&P100"
end_swan = "Swan10D"         #equals the last swan before the swan we are going to predict
start_swan="Swan6"
start = dt(2023,8,28)
end = dt(2024,12,2)
selected_stock_number = 22     #number of stock preselected
start_swan_date = dt(2024,12,2)
end_swan_date = dt(2024,12,5)
w1 = 0.14
w2 = 0.14
MA_days = 30
N = 7 # This parameter is set to help classification method like K-means

# Step 1 - Preselection

In [152]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("All_SwanBeta100.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("All_SwanBeta500.xlsx",index_col=0)

In [153]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

In [154]:
portfolio_beta= portfolio_beta.loc[:,"Swan1":end_swan]
portfolio_beta

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Ticker,,,,,,,,,,
SO,0.735961,-0.130115,-0.265152,-0.138738,0.669546,-0.279809,0.055185,-0.091814,1.173178,0.264469
DUK,0.400895,-0.126448,-0.519440,1.580780,0.771715,-0.322795,0.233695,-0.356252,1.165454,0.534699
NEE,0.579708,0.166846,-0.420656,0.414536,0.503895,0.415370,0.192031,-0.026634,1.062981,0.710514
VZ,0.547390,0.059871,0.109521,0.905674,0.332333,0.064046,0.717548,0.060183,0.430888,1.036945
EXC,0.634396,0.174697,-0.975565,0.519666,0.794675,0.263392,0.605247,-0.123437,1.254518,0.353268
...,...,...,...,...,...,...,...,...,...,...
JPM,1.196511,2.138915,1.576224,1.649018,1.453522,1.887942,1.012903,0.969234,1.348489,1.245036
BAC,0.992028,2.385911,1.731738,0.594107,2.142401,2.667371,1.184258,1.192922,1.509426,1.256673
MS,NaN,NaN,2.109127,1.441339,0.770028,2.806836,1.528749,1.047092,1.605162,0.793801


In [155]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan

In [156]:
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Swan1,1.0,0.003789,0.101347,0.057327,-0.000547,0.083689,0.085222,0.058642,0.007853,-0.097570
Swan2,NaN,1.000000,0.583145,0.360109,0.432200,0.445509,0.039317,0.367065,0.075784,0.484392
Swan3,NaN,NaN,1.000000,0.157751,0.283951,0.423298,0.225053,0.190724,0.075861,0.347336
Swan4,NaN,NaN,NaN,1.000000,-0.002432,0.184384,0.126664,0.249096,0.152797,0.303949
Swan5,NaN,NaN,NaN,NaN,1.000000,0.374752,0.324799,0.214341,0.593319,0.156252
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.265093,0.418944,0.324324,0.247842
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.141909,0.263153,-0.065778
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.148860,0.123835
Swan9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,-0.040622
Swan10D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [157]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

In [158]:
portfolio_beta

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Ticker,,,,,,,,,,
SO,0.735961,-0.130115,-0.265152,-0.138738,0.669546,-0.279809,0.055185,-0.091814,1.173178,0.264469
DUK,0.400895,-0.126448,-0.519440,1.580780,0.771715,-0.322795,0.233695,-0.356252,1.165454,0.534699
NEE,0.579708,0.166846,-0.420656,0.414536,0.503895,0.415370,0.192031,-0.026634,1.062981,0.710514
VZ,0.547390,0.059871,0.109521,0.905674,0.332333,0.064046,0.717548,0.060183,0.430888,1.036945
EXC,0.634396,0.174697,-0.975565,0.519666,0.794675,0.263392,0.605247,-0.123437,1.254518,0.353268
...,...,...,...,...,...,...,...,...,...,...
JPM,1.196511,2.138915,1.576224,1.649018,1.453522,1.887942,1.012903,0.969234,1.348489,1.245036
BAC,0.992028,2.385911,1.731738,0.594107,2.142401,2.667371,1.184258,1.192922,1.509426,1.256673
MS,NaN,NaN,2.109127,1.441339,0.770028,2.806836,1.528749,1.047092,1.605162,0.793801


In [159]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1,n_init=10)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group


# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 1000 trials with davies bouldin index
    for r in range(1,1001):
      Kmean = KMeans(n_clusters=N,random_state=r,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[r]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,5):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <13:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    d = {'stock':[], 'frequency':[]}
    df = pd.DataFrame(data=d)
    for item in sorted_count:
      df.loc[len(df.index)] = [item[0],item[1]]
    df.to_csv('stock_frequency.csv')

    for p in range(0,min(22,len(sorted_count))):
      if sorted_count[p][1]>=1:
        preselected_stock.append(sorted_count[p][0])

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #selected_beta["avg"]=selected_beta["avg"].mean()
    preselected_stock = selected_beta
    print(preselected_stock)


['SO', 'AMT', 'MCD', 'MDLZ', 'PEP', 'PG', 'KO', 'SBUX', 'EXC', 'VZ', 'NEE', 'DUK', 'AAPL', 'MA', 'LMT', 'COST', 'GD', 'V', 'CVS', 'BRK-B', 'AMGN', 'IBM', 'GILD', 'UNH', 'BMY', 'WMT', 'PM', 'MO', 'MRK', 'LLY', 'T', 'JNJ', 'CL', 'SO', 'SBUX', 'MCD', 'MDLZ', 'CL', 'PEP', 'PG', 'AMT', 'DUK', 'EXC', 'KO', 'VZ', 'NEE', 'SO', 'DUK', 'NEE', 'VZ', 'EXC', 'MDLZ', 'SBUX', 'KO', 'PG', 'MCD', 'PEP', 'AMT', 'CL', 'SO', 'DUK', 'NEE', 'VZ', 'EXC', 'MDLZ', 'SBUX', 'KO', 'PG', 'MCD', 'PEP', 'AMT', 'CL', 'SO', 'DUK', 'NEE', 'VZ', 'EXC', 'MDLZ', 'SBUX', 'KO', 'PG', 'MCD', 'PEP', 'AMT', 'CL']
[('AMT', 5), ('CL', 5), ('DUK', 5), ('EXC', 5), ('KO', 5), ('MCD', 5), ('MDLZ', 5), ('NEE', 5), ('PEP', 5), ('PG', 5), ('SBUX', 5), ('SO', 5), ('VZ', 5), ('AAPL', 1), ('AMGN', 1), ('BMY', 1), ('BRK-B', 1), ('COST', 1), ('CVS', 1), ('GD', 1), ('GILD', 1), ('IBM', 1), ('JNJ', 1), ('LLY', 1), ('LMT', 1), ('MA', 1), ('MO', 1), ('MRK', 1), ('PM', 1), ('T', 1), ('UNH', 1), ('V', 1), ('WMT', 1)]
           Swan6     Swan7   

<ipython-input-159-36b17c01c415>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-159-36b17c01c415>:74: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
<ipython-input-159-36b17c01c415>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group

In [160]:
res

array([0.54561192, 1.30817427, 0.4320442 , 0.69227737, 0.91685926,
       1.55979299, 0.77249889])

In [161]:
preselected_stock.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22 entries, AAPL to VZ
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Swan6    22 non-null     float64
 1   Swan7    22 non-null     float64
 2   Swan8    22 non-null     float64
 3   Swan9    22 non-null     float64
 4   Swan10D  22 non-null     float64
 5   Group    22 non-null     int32  
 6   avg      22 non-null     float64
dtypes: float64(6), int32(1)
memory usage: 1.3+ KB


In [162]:
#portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan]*weight

In [163]:
portfolio_beta = pd.read_csv("HoltestimateS11_SP100.csv",index_col=0)
if 'Ticker' in preselected_stock.columns:
    preselected_stock.set_index('Ticker', inplace=True)
preselected_stock['avg'] = preselected_stock.index.map(portfolio_beta['Forecasted Swan11'])

In [164]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock

,Swan6,Swan7,Swan8,Swan9,Swan10D,Group,avg,PR
Ticker,,,,,,,,
AAPL,0.279701,1.582785,1.829198,0.972645,0.869585,4,0.766524,0.233476
AMGN,0.109295,0.843475,0.659977,0.526728,-0.417885,0,-1.208626,2.208626
AMT,0.050751,0.583637,-0.336659,0.882155,1.271482,2,1.312267,-0.312267
BMY,-0.398966,1.296441,1.085743,0.931988,-0.566874,0,-1.841029,2.841029
BRK-B,0.533684,0.870710,0.663086,0.928445,0.449181,0,0.655643,0.344357
CL,-0.045655,0.784583,0.569564,0.632697,0.619560,2,0.865733,0.134267
COST,-0.015789,0.618965,0.918644,0.368083,0.635429,0,0.820532,0.179468
CVS,0.689865,0.720939,1.037474,0.825782,0.524514,4,0.691676,0.308324
DUK,-0.322795,0.233695,-0.356252,1.165454,0.534699,2,1.044985,-0.044985


# Step 2 - Download data and adjuste price based on Swan Beta

In [165]:
#next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
#selected_stock_list = pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
#swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

In [166]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

['AAPL' 'AMGN' 'AMT' 'BMY' 'BRK-B' 'CL' 'COST' 'CVS' 'DUK' 'EXC' 'GD'
 'GILD' 'IBM' 'KO' 'MCD' 'MDLZ' 'NEE' 'PEP' 'PG' 'SBUX' 'SO' 'VZ']


In [167]:
#download data from yahoo finance
yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]


[*********************100%%**********************]  22 of 22 completed


In [168]:
yahooDataFrame

Ticker,AAPL,AMGN,AMT,BMY,BRK-B,CL,COST,CVS,DUK,EXC,...,IBM,KO,MCD,MDLZ,NEE,PEP,PG,SBUX,SO,VZ
Date,,,,,,,,,,,,,,,,,,,,,
2023-08-28,179.075638,247.082748,170.773392,58.083755,355.549988,71.578499,520.269165,63.990948,85.499184,38.280807,...,139.345200,57.842083,273.846130,69.377319,65.110748,173.589920,149.103485,94.056396,65.057205,30.796684
2023-08-29,182.981323,250.058731,173.965164,59.223949,358.290009,71.413261,526.653687,63.886269,85.755623,38.518578,...,139.755554,57.784771,273.759186,69.309456,65.070267,174.244781,149.190750,96.101524,65.477364,31.843130
2023-08-30,186.489487,248.363693,174.712769,58.781063,361.059998,71.782616,526.158813,63.600773,85.432686,38.442486,...,140.146820,57.756123,274.386688,69.396706,64.665436,174.389252,149.355576,96.188744,65.525101,31.797232
2023-08-31,186.708115,246.880508,173.792633,58.093182,360.200012,71.413261,532.970398,62.020973,84.340454,38.157169,...,140.118179,57.144848,272.862976,69.096130,64.385933,172.551422,149.646439,94.444099,64.675247,32.109333
2023-09-01,188.288315,247.236832,173.284637,58.441833,362.459991,71.218849,528.089722,62.496819,83.514145,38.033531,...,141.177444,56.648182,272.659180,67.573799,64.453400,170.029892,149.811295,94.986885,64.188255,31.999178
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-22,229.869995,294.529999,204.309998,58.869999,476.570007,94.919998,964.010010,58.009998,114.699997,38.709999,...,222.970001,63.438839,288.544250,64.400002,76.000000,162.000000,176.279999,102.500000,87.599998,43.150002
2024-11-25,232.869995,294.000000,207.550003,58.779999,477.429993,95.339996,960.890015,60.080002,115.000000,39.130001,...,226.130005,63.895374,294.418915,64.940002,76.639999,163.050003,177.389999,101.839996,88.419998,43.980000
2024-11-26,235.059998,280.010010,208.000000,58.740002,478.559998,96.580002,971.500000,59.009998,117.040001,39.490002,...,228.830002,64.064102,294.558075,64.879997,77.489998,162.160004,179.309998,100.680000,89.330002,44.369999


In [169]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [170]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")

In [171]:
market_drop.to_csv("market_drop.csv")

In [172]:
market_drop

,0
0,0.0000
1,0.0028
2,0.0055
3,0.0083
4,0.0110
...,...
313,0.0000
314,0.0000
315,0.0000
316,0.0000


In [173]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

In [174]:
yahooDataFrame


Ticker,AAPL,AMGN,AMT,BMY,BRK-B,CL,COST,CVS,DUK,EXC,...,IBM,KO,MCD,MDLZ,NEE,PEP,PG,SBUX,SO,VZ
Date,,,,,,,,,,,,,,,,,,,,,
2023-08-28,179.075638,247.082748,170.773392,58.083755,355.549988,71.578499,520.269165,63.990948,85.499184,38.280807,...,139.345200,57.842083,273.846130,69.377319,65.110748,173.589920,149.103485,94.056396,65.057205,30.796684
2023-08-29,182.588596,250.904968,173.325956,59.529242,357.632260,71.240152,525.443705,63.762541,85.504706,38.441038,...,139.919226,57.700016,273.198200,69.162650,64.904589,174.145859,148.868644,95.858395,65.314916,31.757565
2023-08-30,185.703269,250.014677,173.451785,59.376260,359.758002,71.440820,523.784298,63.358822,84.941669,38.290477,...,140.469219,57.589722,273.282226,69.107974,64.342021,174.194778,148.722169,95.710735,65.205774,31.629400
2023-08-31,185.520249,249.357115,171.899716,58.980877,358.239862,70.900115,529.340648,61.664916,83.608938,37.929477,...,140.604609,56.896392,271.205498,68.662295,63.899981,172.261037,148.688709,93.735825,64.199604,31.853574
2023-09-01,186.700711,250.523818,170.783285,59.625358,359.845903,70.540627,523.323262,62.021316,82.554164,37.732747,...,141.826983,56.321766,270.464163,67.011504,63.808692,169.650667,148.540616,94.042813,63.562631,31.661383
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-22,229.869995,294.529999,204.309998,58.869999,476.570007,94.919998,964.010010,58.009998,114.699997,38.709999,...,222.970001,63.438839,288.544250,64.400002,76.000000,162.000000,176.279999,102.500000,87.599998,43.150002
2024-11-25,232.869995,294.000000,207.550003,58.779999,477.429993,95.339996,960.890015,60.080002,115.000000,39.130001,...,226.130005,63.895374,294.418915,64.940002,76.639999,163.050003,177.389999,101.839996,88.419998,43.980000
2024-11-26,235.059998,280.010010,208.000000,58.740002,478.559998,96.580002,971.500000,59.009998,117.040001,39.490002,...,228.830002,64.064102,294.558075,64.879997,77.489998,162.160004,179.309998,100.680000,89.330002,44.369999


In [175]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['AAPL', 'AMGN', 'AMT', 'BMY', 'BRK-B', 'CL', 'COST', 'CVS', 'DUK',
       'EXC', 'GD', 'GILD', 'IBM', 'KO', 'MCD', 'MDLZ', 'NEE', 'PEP',
       'PG', 'SBUX', 'SO', 'VZ'], dtype=object)

# Step3 - MVO

In [176]:
#Set the paramter for MVO process
Portfolio_Date=end

In [177]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

In [178]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)



In [179]:
# Calculate MA
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

In [180]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

In [181]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

In [182]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [183]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

In [184]:
# Read OPT-data
#optdata = pd.read_excel("optdata_SP100_2024_10_01.xlsx",index_col=0)
optdata = pd.read_csv("optdata_SP100_20241202.csv",index_col=0)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index]
R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0

In [185]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
preselected_stock.to_csv("preselected_Stocks_info.csv")
p=FFC.MA_now
EPS_fwd=optimizer.EPS_fwd
EPS_ttm=optimizer.EPS_ttm
fall=FFC.Fall_MDD
FFC.to_csv("WEDataBank.csv")

PE=p/EPS_fwd
PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
PEG[PEG<0]=2000


In [186]:
preselected_stock

,Swan6,Swan7,Swan8,Swan9,Swan10D,Group,avg,PR,Rfwd_adjusted,std_f,SR,PR_SR
Ticker,,,,,,,,,,,,
AAPL,0.279701,1.582785,1.829198,0.972645,0.869585,4,0.766524,0.233476,0.301213,0.238324,1.263878,0.262479
AMGN,0.109295,0.843475,0.659977,0.526728,-0.417885,0,-1.208626,2.208626,0.055603,0.299317,0.185767,0.951933
AMT,0.050751,0.583637,-0.336659,0.882155,1.271482,2,1.312267,-0.312267,0.219806,0.260837,0.842696,-0.286656
BMY,-0.398966,1.296441,1.085743,0.931988,-0.566874,0,-1.841029,2.841029,0.055366,0.324001,0.170883,1.174424
BRK-B,0.533684,0.870710,0.663086,0.928445,0.449181,0,0.655643,0.344357,0.171959,0.154635,1.112026,0.363134
CL,-0.045655,0.784583,0.569564,0.632697,0.619560,2,0.865733,0.134267,0.073035,0.162681,0.448944,0.089963
COST,-0.015789,0.618965,0.918644,0.368083,0.635429,0,0.820532,0.179468,0.235472,0.206862,1.138307,0.191477
CVS,0.689865,0.720939,1.037474,0.825782,0.524514,4,0.691676,0.308324,0.142059,0.337295,0.421171,0.200095
DUK,-0.322795,0.233695,-0.356252,1.165454,0.534699,2,1.044985,-0.044985,0.210036,0.160198,1.311100,-0.051509


In [187]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = Rfwd_adjusted/fall
E_3 = Rfwd_adjusted/PEG
E_4 = Rfwd_adjusted
std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


# Optimization

In [202]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

In [203]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)


OrderedDict([('AAPL', 0.14), ('AMGN', 0.0), ('AMT', 0.0), ('BMY', 0.02475), ('BRK-B', 0.14), ('CL', 0.0), ('COST', 0.14), ('CVS', 0.00667), ('DUK', 0.14), ('EXC', 0.0), ('GD', 0.0), ('GILD', 0.13364), ('IBM', 0.03242), ('KO', 0.0), ('MCD', 0.0), ('MDLZ', 0.0), ('NEE', 0.0), ('PEP', 0.0), ('PG', 0.0), ('SBUX', 0.0), ('SO', 0.14), ('VZ', 0.10252)])
Expected annual return: 2.3%
Annual volatility: 10.2%
Sharpe Ratio: 0.12


/usr/local/lib/python3.10/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


(0.022526469893874027, 0.1021216009897684, 0.1226622944848765)

In [204]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

In [205]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1
print("new_sharpe:",sharpe_ratio1)

new_sharpe: 0.12266229447113533


weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

sd2 = np.sqrt(weight2@cov@weight2.T)
R2 = weight2.T@E_2/100
sharpe_ratio2 = (R2-0.01)/sd2
print("old_sharpe:",sharpe_ratio2)

In [206]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.03]
weight.weight = weight.weight/sum(weight.weight)
# weight = pd.read_csv("weights.csv",index_col=0)
weight.to_csv("weights.csv")

See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference


In [207]:
if stock_type == "S&P100":
    OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

elif stock_type == "S&P500":
    OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed


In [208]:
stock_price.to_csv("tradeinfo.csv")
stock_price

Ticker,AAPL,BRK-B,COST,DUK,GILD,IBM,SO,VZ
Date,,,,,,,,
2024-12-02,239.589996,477.329987,975.000000,115.180000,94.019997,227.389999,87.769997,43.849998
2024-12-03,242.649994,470.190002,982.150024,113.639999,92.489998,229.000000,86.260002,43.830002
2024-12-04,243.009995,468.320007,990.919983,113.279999,91.589996,233.490005,85.849998,42.520000


In [209]:
print(stock_price.head(1))
weight


Ticker            AAPL       BRK-B   COST     DUK       GILD         IBM  \
Date                                                                       
2024-12-02  239.589996  477.329987  975.0  115.18  94.019997  227.389999   

Ticker             SO         VZ  
Date                              
2024-12-02  87.769997  43.849998  


,weight
AAPL,0.144541
BRK-B,0.144541
COST,0.144541
DUK,0.144541
GILD,0.137975
IBM,0.033472
SO,0.144541
VZ,0.105846


In [210]:
rest=1
extra=0
for i, row in weight.iterrows():
  if row['weight']>0.14:
    rest=rest-row['weight']
    extra=extra+(row['weight']-0.14)
    row['weight']=0.14

for i,row in weight.iterrows():
  if row['weight']<0.135:
    row['weight']=row['weight']+row['weight']/rest*extra
    a=1

weight.to_csv("weights.csv")
weight

,weight
AAPL,0.140000
BRK-B,0.140000
COST,0.140000
DUK,0.140000
GILD,0.137975
IBM,0.036213
SO,0.140000
VZ,0.114513


In [211]:
weight = pd.read_csv("weights.csv",index_col=0)
injected_result = pd.DataFrame(index = stock_price.index)
S1 = np.array(1398557.4210899*weight.T)/np.array(stock_price.iloc[0,:]).T
weight

,weight
AAPL,0.140000
BRK-B,0.140000
COST,0.140000
DUK,0.140000
GILD,0.137975
IBM,0.036213
SO,0.140000
VZ,0.114513


In [212]:
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(1398557.4210899*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 1258120.96018462/float(OEX[0])
injected_result["oex"]=S2*OEX
injected_result.to_csv("Injected_result.csv")


<ipython-input-212-4aef90511ae0>:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 1258120.96018462/float(OEX[0])


In [213]:
injected_result

,portfolio_value,oex
Date,,
2024-12-02,1.382755e+06,1.258121e+06
2024-12-03,1.374922e+06,1.260860e+06
2024-12-04,1.369052e+06,1.271052e+06


In [214]:
allocation=pd.read_excel('allocation weight.xlsx',index_col='stock selected')
dfMVOstock=deepcopy(weight)
dfMVOstock['PR_SR']=''
dfMVOstock['SR']=''
for i in range(len(weight)):
    dfMVOstock.iloc[i,1]=preselected_stock.loc[weight.index[i],'PR_SR']
    dfMVOstock.iloc[i,2]=preselected_stock.loc[weight.index[i],'SR']
dfMVOstock_sort=dfMVOstock.sort_values(by='PR_SR',ascending=False)
postallocationweight=post_allocation(dfMVOstock_sort)
postallocationweight


<ipython-input-16-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<ipython-input-16-2b9c13413cb0>:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfMVOweight.iloc[i,5]=allocationweight[0]
<ipython-input-16-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<i

,weight,PR_SR,SR,equal,proportion,adjusted proportion,direct
IBM,0.036213,1.583797,1.247059,0.125,0.406325,0.17,0.17
GILD,0.137975,1.382827,1.545769,0.125,0.354766,0.17,0.17
BRK-B,0.140000,0.363134,1.112026,0.125,0.093162,0.163344,0.17
AAPL,0.140000,0.262479,1.263878,0.125,0.067339,0.137521,0.17
COST,0.140000,0.191477,1.138307,0.125,0.049124,0.119306,0.17
SO,0.140000,0.133739,1.377825,0.125,0.034311,0.104493,0.05
VZ,0.114513,0.031913,0.626133,0.125,0.008187,0.078369,0.05
DUK,0.140000,-0.051509,1.3111,0.125,-0.013215,0.056967,0.05


In [215]:
#draw type E graph for postallocation
MVOweight=postallocationweight['weight']
equalweight=postallocationweight['equal']
proportionweight=postallocationweight['adjusted proportion']
directweight=postallocationweight['direct']
output(MVOweight).to_csv('no_post_allocation_result.csv')
output(equalweight).to_csv('equal_weight_result.csv')
output(proportionweight).to_csv('proportional_weight_result.csv')
output(directweight).to_csv('direct_weight_result.csv')

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed
<ipython-input-17-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed
<ipython-input-17-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 comple

#